In [3]:
import numpy as np
import scipy as sp
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
from scipy.special import erf
import lmfit
from lmfit import Model, Parameters
from lmfit.model import save_modelresult, load_modelresult
import emcee
import glob
from scipy.ndimage import gaussian_filter1d
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
%matplotlib widget

# Define Fitting functions
define a the function used to fit the data here. first argument should be q (or x-axis) and following arguments are fitted variables. Return should be intensity (or y-axis) that will be used to calculate goodness of fit

*Don't forget to update func_dict={} at the bottom of the cell

In [4]:
def selective_gaussian_filter(x, y, threshold=0.05, sigma=2):
    """Apply a Gaussian filter to y values where x is greater than a threshold"""
    smoothed_y = y.copy()  # Start with the original data
    mask = x > threshold  # Create a mask for elements where x > threshold
    smoothed_y[mask] = gaussian_filter1d(y[mask], sigma=sigma, mode='nearest')
    return smoothed_y
    
def lorentz_peak(x, A, sigma, x0):
    """
    Calculate the Lorentzian peak function.

    Parameters:
    x : array_like, The independent variable where the Lorentz peak is evaluated.
    A : float, Amplitude of the Lorentzian peak (integrated scattering intensity).
    sigma : float, Half-width at half-maximum (HWHM) of the Lorentzian peak.
    x0 : float, Position of the center of the Lorentzian peak.

    Returns:
    array_like, Values of the Lorentzian function at each x.
    """
    return (A / np.pi) * (sigma / ((x - x0)**2 + sigma**2))

def porod(q, c, p):
    """
    Calculates the scattering intensity based on porod exponential.
    
    Parameters:
    q : array_like, Array of q values.
    c : float, porod scale
    p : float, porod exponent

    Returns:
    array_like
        Scattering intensity for each q value.
    """    
    # Calculate the total intensity I(q)
    Iq = np.zeros_like(q)
    # porod region term
    Iq = c*(1 / q)**p

    return Iq

def level_guinier_porod(q, g, b, rg, rgco, p):
    """
    Calculates the scattering intensity based on the provided parameters.
    this covers a single level, must iterate for multiple levels.
    https://www.sasview.org/docs/user/models/unified_power_Rg.html

    Parameters:
    q : array_like, Array of q values.
    g : float, G value
    b : float, B value
    rg : float, radius of gyration
    rgco : float, radius of gyration cut off from next level
    p : float, porod exponent

    Returns:
    array_like
        Scattering intensity for each q value.
    """    
    # Calculate the total intensity I(q)
    Iq = np.zeros_like(q)
    # Calculate q* for the current level
    qi_star = q * erf(q * rg / np.sqrt(6))**-3
    # Guinier region term
    term1 = g * np.exp(-q**2 * rg**2 / 3)
    # porod region term
    term2 = b * np.exp(-q**2 * rgco**2 / 3) * (1 / qi_star)**p
    # Sum up contributions from this level to total I(q)
    Iq += term1 + term2

    return Iq
    
def two_level_unified_bkg(q, g1, g2, b1, b2, rg1, rg2, p1, p2, bkg):
    """
    Calculates the scattering intensity based on the provided parameters.
    this covers a three levels, wraps level_guinier_porod().
    https://www.sasview.org/docs/user/models/unified_power_Rg.html

    Parameters (for each level):
    q : array_like, Array of q values.
    g : float, G value
    b : float, B value
    rg : float, radius of gyration
    p : float, porod exponent

    Returns:
    array_like
        Scattering intensity for each q value.
    """
    Iq_1 = level_guinier_porod(q, g1, b1, rg1, rg2, p1)
    Iq_2 = level_guinier_porod(q, g2, b2, rg2, 0, p2)
    bkg_vals = np.full_like(q, bkg)
    
    return Iq_1 + Iq_2 + bkg_vals

def three_level_unified(q, g1, g2, g3, b1, b2, b3, rg1, rg2, rg3, p1, p2, p3):
    """
    Calculates the scattering intensity based on the provided parameters.
    this covers a three levels, wraps level_guinier_porod().
    https://www.sasview.org/docs/user/models/unified_power_Rg.html

    Parameters (for each level):
    q : array_like, Array of q values.
    g : float, G value
    b : float, B value
    rg : float, radius of gyration
    p : float, porod exponent

    Returns:
    array_like
        Scattering intensity for each q value.
    """
    Iq_1 = level_guinier_porod(q, g1, b1, rg1, rg2, p1)
    Iq_2 = level_guinier_porod(q, g2, b2, rg2, rg3, p2)
    Iq_3 = level_guinier_porod(q, g3, b3, rg3, 0, p3)
    
    return Iq_1 + Iq_2 + Iq_3

def three_level_unified_peak(q, g1, g2, g3, b1, b2, b3, rg1, rg2, rg3, p1, p2, p3, A, sigma, x0):
    """
    Calculates the scattering intensity based on the provided parameters.
    this covers a three guinier porod levels and a lorentz peak.
    https://www.sasview.org/docs/user/models/unified_power_Rg.html

    q : array_like, Array of q values.
    
    unified model parameters (for each level):
    g : float, G value
    b : float, B value
    rg : float, radius of gyration
    p : float, porod exponent

    lorentz peak parameters:
    A : float, Amplitude of the Lorentzian peak (integrated scattering intensity).
    sigma : float, Half-width at half-maximum (HWHM) of the Lorentzian peak.
    x0 : float, Position of the center of the Lorentzian peak.

    Returns:
    Iq: array_like, Scattering intensity for each q value.
    """
    Iq_1 = level_guinier_porod(q, g1, b1, rg1, rg2, p1)
    Iq_2 = level_guinier_porod(q, g2, b2, rg2, rg3, p2)
    Iq_3 = level_guinier_porod(q, g3, b3, rg3, 0, p3)
    Iq_peak = lorentz_peak(q, A, sigma, x0)

    
    return Iq_1 + Iq_2 + Iq_3 + Iq_peak



func_dict = {
    'three_level_unified_peak': three_level_unified_peak,
    'three_level_unified': three_level_unified,
    'level_guinier_porod': level_guinier_porod,
    'lorentz_peak': lorentz_peak,
    'two_level_unified_bkg':two_level_unified_bkg
             }

# Fit Data to model
Here you will fit the data to a selected model. 
- For first run both load_prev_vals and load_from_file should be false. These are used to initialize parameter start values from the previous fit or some saved fit respectively. 
- Make sure to define the directory where your experimental data is stored, the directory where you would like to save fit output parameters/plots and the directory where you would like to load previous fits 
- Below this is a block which I have used to find load and trim my experimental data. exp_data should only be the intensity (y-axis) values used for fitting. The q (x-axis) values are loaded in later as q_vals 



In [ ]:
load_prev_vals = False
load_from_file = True
#define experimental paths
exp_dirr = '/Users/Thomas2/Library/CloudStorage/OneDrive-UCB-O365/Desktop/Research_Stuff/Amnahir_SAXS/April2024_ssrl/analysis_10frames_May22/subtracted_vfcalc/'
save_dirr = '/Users/Thomas2/Library/CloudStorage/OneDrive-UCB-O365/Desktop/Research_Stuff/Amnahir_SAXS/April2024_ssrl/analysis_10frames_May22/fitting_vfcalc_dpp3_rg1_vary_b_fixed/'
load_dirr = '/Users/Thomas2/Library/CloudStorage/OneDrive-UCB-O365/Desktop/Research_Stuff/Amnahir_SAXS/April2024_ssrl/analysis_10frames_May22/fitting_vfcalc_dpp3_rg1_vary_b_fixed/'
#load and trim experimental intensity values
exp_sample = 'DPP3'
exp_temp = '25C'
exp_file = glob.glob(f'{exp_dirr}linecut_{exp_sample}_{exp_temp}_S*_sub.txt')[0]
exp_data = np.loadtxt(exp_file)
exp_data = exp_data[1:,:]
exp_data = exp_data[exp_data[:,0]<0.24]
exp_data = exp_data[exp_data[:,1]>0.005]
exp_data[:,1] = selective_gaussian_filter(exp_data[:,0], exp_data[:,1], threshold=0.03, sigma=3)

# Define fitting function being used here
model = Model(three_level_unified_peak)

# # Create a Parameters object and set initial values and constraints
#variables must match arguments for fitting function
#variables are defined as list ['variable_name', start_value, lower_bound, upper_bound, Allow to vary?(True/False)]
q_vals = exp_data[:,0]
g1 = ['g1', 0,0,1e4,False] 
g2 = ['g2', 7,1e-6,10,True] 
g3 = ['g3', 1.5,1e-6,10,True] 
b1 = ['b1', 3.6639e-05,1e-7,1e-2,True] 
b2 = ['b2', 0,0,1,False] 
b3 = ['b3', 0,0,1,False] 
rg1 = ['rg1', 500,0,1000,False] 
rg2 = ['rg2', 66,0,100,False] 
rg3 = ['rg3', 23,0,30,False] 
p1 = ['p1', 3,2.8,3.1,True]
p2 = ['p2', 1,0,4,False]
p3 = ['p3', 1,0,4,False]
A = ['A', 0.02,0,100,True]
sigma = ['sigma', 0.05,0,100,True] 
x0 = ['x0', 0.17,0.15,0.25,True]

#variable_list is an array of variable names, values, bounds
variable_list = [g1, g2, g3, b1, b2, b3, rg1, rg2, rg3, p1, p2, p3, A, sigma, x0]

#Here variable parameters are reset from a file or from previous fit output 
if load_from_file:
    result = load_modelresult(f'{load_dirr}fit_results_{exp_sample}_{exp_temp}.sav', funcdefs=func_dict)
if load_prev_vals or load_from_file:
    for var in variable_list:
        var[1] = result.params[var[0]].value
        var[2] = result.params[var[0]].min
        var[3] = result.params[var[0]].max
        var[4] = result.params[var[0]].vary

#populate params with variable parameters
params = Parameters()
for var in variable_list:
    params.add(var[0], value=var[1], min=var[2], max=var[3], vary=var[4])

# This defines the weighting for loss function. For data intenisty that spans many orders of magnitude 1/intensity weighting is good. 
# None can be used for no weighting
weights = 1/exp_data[:,1]

# # Perform the fit
result = model.fit(exp_data[:,1], params, q=q_vals, weights=weights, method='leastsq')
# sometimes different fit algorthms can be useful like differential evolution or nelder 
#method='differential_evolution'#method='nelder'

#Plot fit
fig, ax1 = subplots(1,1)
ax1.plot(exp_data[:,0], exp_data[:,1], label='data', color='gray', linestyle='None', marker='.')
ax1.plot(q_vals, result.best_fit, label='fit', color = 'red', linestyle='--')
#log log plot. comment these lines out for linear plot axes
ax1.set_xscale('log')
ax1.set_yscale('log')

In [250]:
#if you like the fit save it along with the fit curve as a .npy
save_modelresult(result, f'{save_dirr}fit_results_{exp_sample}_{exp_temp}.sav')
np.save(f'{save_dirr}q_vals_{exp_sample}_{exp_temp}.npy', q_vals)

In [ ]:
# result = load_modelresult(f'{load_dirr}fit_results_{exp_sample}_{exp_temp}.sav', funcdefs=func_dict)
print(result.fit_report())

# Plot and save individual or aggregated fits/data

- just including this if it is useful. This is somewhat specified for my dataset but demonstrates how you can quickly make some bar charts for fitted parameters 

In [314]:
exp_dirr = '/Users/Thomas2/Library/CloudStorage/OneDrive-UCB-O365/Desktop/Research_Stuff/Amnahir_SAXS/April2024_ssrl/analysis_10frames_May22/subtracted_vfcalc/'
save_dirr = '/Users/Thomas2/Library/CloudStorage/OneDrive-UCB-O365/Desktop/Research_Stuff/Amnahir_SAXS/April2024_ssrl/analysis_10frames_May22/fitting_vfcalc_sebs10/'
exp_sample = 'sebs10'
exp_temps = ['25C', '40C', '50C', '60C', '70C', '80C', '90C']
for exp_temp in exp_temps:
    exp_file = glob.glob(f'{exp_dirr}linecut_{exp_sample}_{exp_temp}_S*_sub.txt')[0]
    exp_data = np.loadtxt(exp_file)
    exp_data = exp_data[exp_data[:,0]<0.24]
    exp_data = exp_data[exp_data[:,1]>0.005]
    exp_data[:,1] = selective_gaussian_filter(exp_data[:,0], exp_data[:,1], threshold=0.03, sigma=3)
    result = load_modelresult(f'{save_dirr}fit_results_{exp_sample}_{exp_temp}.sav', funcdefs=func_dict)
    q_vals = np.load(f'{save_dirr}q_vals_{exp_sample}_{exp_temp}.npy')
    
    fig, ax1 = subplots(1,1)
    ax1.plot(exp_data[:,0], exp_data[:,1], label='data', color='gray', linestyle='None', marker='.')
    ax1.plot(q_vals, result.best_fit, label='fit', color = 'red', linestyle='--')
    
    fontsize = 14
    ax1.legend(prop={'size': fontsize})
    ax1.tick_params(axis='x', which='major', labelsize=fontsize)
    ax1.tick_params(axis='y', which='major', labelsize=fontsize)
    ax1.set_xlabel('q ($\AA^{-1}$)', size=fontsize)
    ax1.set_ylabel('Intensity (arb. units)', size=fontsize)
    ax1.set_title(f'{exp_sample} at {exp_temp}')
    ax1.set_xscale('log')
    ax1.set_yscale('log')
    plt.tight_layout()
    
    plt.savefig(f'{save_dirr}fit_results_{exp_sample}_{exp_temp}.png', dpi=300)
    plt.close('all')

In [ ]:
exp_dirr = '/Users/Thomas2/Library/CloudStorage/OneDrive-UCB-O365/Desktop/Research_Stuff/Amnahir_SAXS/April2024_ssrl/analysis_10frames_May22/subtracted_vfcalc/'
save_dirr = '/Users/Thomas2/Library/CloudStorage/OneDrive-UCB-O365/Desktop/Research_Stuff/Amnahir_SAXS/April2024_ssrl/analysis_10frames_May22/fitting_vfcalc_sebs10/'
exp_sample = 'sebs10'
exp_temps = ['25C', '40C', '50C', '60C', '70C', '80C', '90C']

colors = plt.cm.plasma(np.linspace(0, 0.7, len(exp_temps)))
fig, ax1 = subplots(1,1)
for i, exp_temp in enumerate(exp_temps):
    exp_file = glob.glob(f'{exp_dirr}linecut_{exp_sample}_{exp_temp}_S*_sub.txt')[0]
    exp_data = np.loadtxt(exp_file)
    exp_data = exp_data[exp_data[:,0]<0.24]
    exp_data = exp_data[exp_data[:,1]>0.005]
    exp_data[:,1] = selective_gaussian_filter(exp_data[:,0], exp_data[:,1], threshold=0.03, sigma=3)
    result = load_modelresult(f'{save_dirr}fit_results_{exp_sample}_{exp_temp}.sav', funcdefs=func_dict)
    q_vals = np.load(f'{save_dirr}q_vals_{exp_sample}_{exp_temp}.npy')
    if i == 0:
        ax1.plot(exp_data[:,0], exp_data[:,1], label='data', color=colors[i], linestyle='None', marker='.', alpha=0.1)
        ax1.plot(q_vals, result.best_fit, label='fit', color=colors[i], linestyle='--', linewidth=2)
    else:
        ax1.plot(exp_data[:,0], exp_data[:,1], color=colors[i], linestyle='None', marker='.', alpha=0.1)
        ax1.plot(q_vals, result.best_fit, color=colors[i], linestyle='--', linewidth=2)
    
fontsize = 13

# Adding a colorbar
norm = Normalize(vmin=25, vmax=90)# Assuming temperature ranges from 25C to 90C
sm = ScalarMappable(norm=norm, cmap='plasma')
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax1, orientation='vertical', fraction=0.046, pad=0.02)
cbar.set_label('Temperature (°C)', size=fontsize)
cbar.ax.tick_params(labelsize=fontsize)

# ax1.legend(prop={'size': fontsize})
ax1.tick_params(axis='x', which='major', labelsize=fontsize)
ax1.tick_params(axis='y', which='major', labelsize=fontsize)
ax1.set_xlabel('q ($\AA^{-1}$)', size=fontsize)
ax1.set_ylabel('Intensity (arb. units)', size=fontsize)
ax1.set_xscale('log')
ax1.set_yscale('log')
plt.tight_layout()
    
# plt.savefig(f'{save_dirr}fit_results_{exp_sample}_fitsonly.png', dpi=300)

# Aggregate Results for a dataset

In [7]:
exp_dirr = '/Users/Thomas2/Library/CloudStorage/OneDrive-UCB-O365/Desktop/Research_Stuff/Amnahir_SAXS/April2024_ssrl/analysis_10frames_May22/subtracted_vfcalc/'
save_dirr = '/Users/Thomas2/Library/CloudStorage/OneDrive-UCB-O365/Desktop/Research_Stuff/Amnahir_SAXS/April2024_ssrl/analysis_10frames_May22/fitting_vfcalc_dpp3_rg1_vary_b_fixed/'
exp_sample = 'DPP3'
exp_temps = ['25C', '40C', '50C', '60C', '70C', '80C', '90C']
exp_temp_ints = [25, 40, 50, 60, 70, 80, 90]
peak_fwhm = []
peak_center = []
peak_amp = []
g1 = []
g2 = []
g3 = []
rg3 = []
rg2 = []
rg1 = []
b1 = []
p1 = []
b2 = []
p2 = []
b3 = []
p3 = []
for i, exp_temp in enumerate(exp_temps):
    result = load_modelresult(f'{save_dirr}fit_results_{exp_sample}_{exp_temp}.sav', funcdefs=func_dict)
    peak_fwhm.append(2*result.params['sigma'].value)
    peak_center.append(result.params['x0'].value)
    peak_amp.append(result.params['A'].value)
    g1.append(result.params['g1'].value)
    g2.append(result.params['g2'].value)
    g3.append(result.params['g3'].value)
    rg1.append(result.params['rg1'].value)
    rg2.append(result.params['rg2'].value)
    rg3.append(result.params['rg3'].value)
    b1.append(result.params['b1'].value)
    p1.append(result.params['p1'].value)
    b2.append(result.params['b2'].value)
    p2.append(result.params['p2'].value)
    b3.append(result.params['b3'].value)
    p3.append(result.params['p3'].value)

In [ ]:
x = np.arange(len(exp_temps))  # the label locations
width = 0.25  # the width of the bars
# Plot setup
fig, ax1 = plt.subplots()

# Bar plot for g1
rects1 = ax1.bar(x - width, g1, width, label='g1', color='b')
ax1.set_ylabel('g1 values', color='b')
ax1.tick_params(axis='y', labelcolor='b')
ax1.set_title('Comparison of g1, g2, and g3 by Temperature')
ax1.set_xticks(x)
ax1.set_xticklabels(exp_temps)

# Create ax2 for g2 with a second y-axis
ax2 = ax1.twinx()
rects2 = ax2.bar(x, g2, width, label='g2', color='r')
ax2.set_ylabel('g2 values', color='r')
ax2.tick_params(axis='y', labelcolor='r')

# Create ax3 for g3 with a third y-axis
ax3 = ax1.twinx()
# Offset the third axis to the right
ax3.spines['right'].set_position(('outward', 60))
rects3 = ax3.bar(x + width, g3, width, label='g3', color='g')
ax3.set_ylabel('g3 values', color='g')
ax3.tick_params(axis='y', labelcolor='g')

plt.tight_layout()

In [ ]:
np.mean(peak_fwhm)

In [ ]:
x = np.arange(len(exp_temps))  # the label locations
width = 0.25  # the width of the bars
# Plot setup
fig, ax1 = plt.subplots()

# g1 = np.asarray(g1)
# g2 = np.asarray(g2)
# g3 = np.asarray(g3)

# g3_fraction = g3/(g1+g2+g3)

# Bar plot for g1
rects1 = ax1.bar(x, peak_fwhm, width, label='rg1', color='orange')
ax1.set_ylabel('peak FWHM (1/Å)', color='black')
# ax1.tick_params(axis='y', labelcolor='b')
ax1.set_xticks(x)
ax1.set_xticklabels(exp_temps)
# ax1.set_yscale('log')

In [ ]:
x = np.arange(len(exp_temps))  # the label locations
width = 0.25  # the width of the bars
# Plot setup
fig, ax1 = plt.subplots()
# Create ax2 for g2 with a second y-axis
rects2 = ax1.bar(x-width/2, g2, width, label='g2', color='r')
ax1.set_ylabel('g2 values', color='r')
ax1.tick_params(axis='y', labelcolor='r')

# Create ax3 for g3 with a third y-axis
ax2 = ax1.twinx()
# Offset the third axis to the right
# ax2.spines['right'].set_position(('outward', 60))
rects3 = ax2.bar(x + width/2, g3, width, label='g3', color='g')
ax2.set_ylabel('g3 values', color='g')
ax2.tick_params(axis='y', labelcolor='g')

plt.tight_layout()

In [ ]:
np.mean(rg2)